# Semana 6 — Soluciones M3: Wide Transformations
**Bootcamp:** Fundamentos de Ingeniería de Datos — Databricks SQL

**Instructor:** Luciano Argolo | [lucianoargolo.com](https://lucianoargolo.com)

**Scope:** groupBy+agg, Exchange/shuffle, JOINs (simple + star schema), Window Functions (row_number, rank, lag), df.write a Delta, 3 versiones de query compleja.

---
## Setup: Leer tablas Gold
---

In [0]:
from pyspark.sql.functions import col, avg, count, min as spark_min, max as spark_max, round as spark_round, desc, row_number, rank, lag, abs as spark_abs
from pyspark.sql.window import Window

df_fact = spark.table("bootcamp.gold.fact_propiedades")
df_zona = spark.table("bootcamp.gold.dim_zona")
df_caract = spark.table("bootcamp.gold.dim_caracteristicas")
df_tipo_op = spark.table("bootcamp.gold.dim_tipo_operacion")
df_orient = spark.table("bootcamp.gold.dim_orientacion")

print("=== fact_propiedades ===")
df_fact.printSchema()
print(f"Registros: {df_fact.count():,}")

print("\n=== dim_zona ===")
df_zona.printSchema()

---
## Parte 1: Agregaciones y JOINs
---

### Ejercicio 3.1: groupBy + agg — GUIDED

**Objetivo:** Aplicar agregaciones wide y entender el shuffle.

In [0]:
# groupBy es wide: requiere shuffle para agrupar datos de distintas particiones
df_por_zona = (
    df_fact
    .groupBy("zona_id")
    .agg(
        count("*").alias("cantidad"),
        spark_round(avg("precio"), 2).alias("precio_promedio"),
        spark_round(spark_min("precio"), 2).alias("precio_minimo"),
        spark_round(spark_max("precio"), 2).alias("precio_maximo")
    )
    .orderBy(desc("cantidad"))
)
df_por_zona.show(10)

In [0]:
# .explain() — Exchange = shuffle. groupBy necesita juntar registros de la misma zona_id
df_por_zona.explain()

### Ejercicio 3.2: JOIN simple — GUIDED

**Objetivo:** Hacer JOIN entre fact y una dimensión para obtener nombres legibles.

In [0]:
# JOIN fact + dim_zona + dim_tipo_operacion (wide transformation)
df_con_zona = (
    df_fact
    .join(df_zona, df_fact.zona_id == df_zona.zona_id, "inner")
    .join(df_tipo_op, df_fact.tipo_operacion_id == df_tipo_op.tipo_operacion_id, "inner")
    .select(
        df_zona.partido,
        df_zona.region,
        df_fact.precio,
        df_fact.precio_por_m2,
        df_tipo_op.moneda
    )
)
df_con_zona.show(10)

### Ejercicio 3.3: JOIN star schema completo — INDEPENDENT

**Objetivo:** Reconstruir el star schema con JOINs múltiples en PySpark.

In [0]:
# JOIN completo del star schema + agregación
df_resumen = (
    df_fact
    .join(df_zona, df_fact.zona_id == df_zona.zona_id, "inner")
    .join(df_tipo_op, df_fact.tipo_operacion_id == df_tipo_op.tipo_operacion_id, "inner")
    .join(df_caract, df_fact.caracteristicas_id == df_caract.caracteristicas_id, "inner")
    .groupBy(df_zona.partido, df_tipo_op.tipo_operacion)
    .agg(
        count("*").alias("cantidad"),
        spark_round(avg(df_fact.precio), 2).alias("precio_promedio"),
        spark_round(avg(df_fact.precio_por_m2), 2).alias("precio_m2_promedio")
    )
    .filter(col("cantidad") > 50)
    .orderBy(desc("precio_promedio"))
)
df_resumen.show(20)

---
## Parte 2: Window Functions
---

### Ejercicio 3.4: Window Functions — ranking — INDEPENDENT

**Objetivo:** Aplicar Window Functions para rankings dentro de particiones.

In [0]:
# Preparar DF con partido y precio (JOIN fact + dims)
df_venta_usd = (
    df_fact
    .join(df_zona, df_fact.zona_id == df_zona.zona_id, "inner")
    .join(df_tipo_op, df_fact.tipo_operacion_id == df_tipo_op.tipo_operacion_id, "inner")
    .filter(df_tipo_op.tipo_operacion == "Venta")
    .filter(df_tipo_op.moneda == "USD")
    .select(df_zona.partido, df_fact.precio, df_fact.precio_por_m2)
)

# Window: ranking dentro de cada partido
window_spec = Window.partitionBy("partido").orderBy(desc("precio"))

df_ranking = (
    df_venta_usd
    .withColumn("row_num", row_number().over(window_spec))
    .withColumn("rank", rank().over(window_spec))
    .filter(col("row_num") <= 3)
    .orderBy("partido", "row_num")
)
df_ranking.show(30)

### Ejercicio 3.5: lag() — diferencia entre propiedades — INDEPENDENT

**Objetivo:** Usar `lag()` para comparar filas consecutivas dentro de una ventana.


In [0]:
# lag() — comparar precio con la fila anterior en la ventana
window_lag = Window.partitionBy("partido").orderBy(desc("precio"))

df_diferencias = (
    df_venta_usd
    .withColumn("precio_siguiente", lag("precio", 1).over(window_lag))
    .withColumn("diferencia", col("precio") - col("precio_siguiente"))
    .filter(col("precio_siguiente").isNotNull())
    .select("partido", "precio", "precio_siguiente", "diferencia")
    .orderBy(spark_abs(col("diferencia")).desc())
)
df_diferencias.show(10)

---
## Parte 3: Persistencia y comparación de planes
---

### Ejercicio 3.6: Persistir a Delta — INDEPENDENT

**Objetivo:** Escribir resultados a una tabla Delta.

In [0]:
# df.write — persistir resultado a Delta
df_resumen.write.format("delta").mode("overwrite").saveAsTable("bootcamp.gold.resumen_zona_tipo_s6")

# Verificar que se creó
spark.table("bootcamp.gold.resumen_zona_tipo_s6").show(10)

### Ejercicio 3.7: spark.sql() vs DataFrame API (complejo) — INDEPENDENT

**Objetivo:** Comparar planes para queries complejas con múltiples JOINs.

In [0]:
# spark.sql() — misma query del ejercicio 3.3
resultado_sql = spark.sql("""
    SELECT 
        dz.partido,
        dto.tipo_operacion,
        COUNT(*) as cantidad,
        ROUND(AVG(fp.precio), 2) as precio_promedio,
        ROUND(AVG(fp.precio_por_m2), 2) as precio_m2_promedio
    FROM bootcamp.gold.fact_propiedades fp
    JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
    JOIN bootcamp.gold.dim_tipo_operacion dto ON fp.tipo_operacion_id = dto.tipo_operacion_id
    JOIN bootcamp.gold.dim_caracteristicas dc ON fp.caracteristicas_id = dc.caracteristicas_id
    GROUP BY dz.partido, dto.tipo_operacion
    HAVING COUNT(*) > 50
    ORDER BY precio_promedio DESC
""")
resultado_sql.show(20)

In [0]:
# .explain() en ambas versiones — mismo plan
print("=== DataFrame API ===")
df_resumen.explain()

print("\n=== spark.sql() ===")
resultado_sql.explain()

In [0]:
%sql
-- Versión SQL puro para comparar
SELECT 
    dz.partido,
    dto.tipo_operacion,
    COUNT(*) as cantidad,
    ROUND(AVG(fp.precio), 2) as precio_promedio,
    ROUND(AVG(fp.precio_por_m2), 2) as precio_m2_promedio
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
JOIN bootcamp.gold.dim_tipo_operacion dto ON fp.tipo_operacion_id = dto.tipo_operacion_id
JOIN bootcamp.gold.dim_caracteristicas dc ON fp.caracteristicas_id = dc.caracteristicas_id
GROUP BY dz.partido, dto.tipo_operacion
HAVING COUNT(*) > 50
ORDER BY precio_promedio DESC;

---
### Reflexión M3

Respondé con tus palabras:

1. **¿Cuál es la diferencia práctica entre una narrow y una wide transformation?** ¿Cuál es más costosa y por qué?
2. **¿Qué es un shuffle (Exchange) y por qué deberías minimizarlos?** ¿Cuántos shuffles genera un pipeline con 3 JOINs + 1 groupBy?